# IAN 603 · Module 11
# Advanced SQL: CTEs & Window Functions

This notebook is the hands-on, step-by-step companion for **Module 11: Advanced SQL — CTEs & Window Functions**. It uses small tables and intermediate results to show not only *what* the SQL does, but *how the data changes at each stage*.

### Main ideas
- Common Table Expressions (CTEs)
- Chained CTEs
- Recursive CTEs
- Window functions
- `PARTITION BY`
- Window `ORDER BY`
- `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`
- `LAG()` and `LEAD()`
- Running totals and moving windows
- Window frames
- `GROUP BY` vs. `PARTITION BY`
- Top-N-per-group patterns
- Readmission interval analysis

> **Important:** This notebook uses PostgreSQL. Your PostgreSQL server must be running and accessible from the machine running the notebook.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module11.html



## Learning objectives

By the end of this notebook, you should be able to:

1. Use one or more CTEs to organize a complex query into readable stages.
2. Explain the anchor and recursive terms in a recursive CTE.
3. Explain why a window function can analyze related rows without collapsing the current row.
4. Choose between `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()` based on tie behavior.
5. Use `LAG()` and `LEAD()` for previous-event and next-event questions.
6. Write explicit window frames for running and moving calculations.
7. Explain the difference between `GROUP BY` and `PARTITION BY`.
8. Track the **grain** of each intermediate result.

## 0. Install required Python packages

Run this only if the packages are not already installed. `ipywidgets` is used for the pgAdmin-style connection panel.

In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 2. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 3. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()

### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 4. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")

# Part I · Create the Module 11 teaching data

The examples use five small tables.

| Table | Grain | Main purpose |
|---|---|---|
| `patients` | one row per patient | patient features |
| `admissions` | one row per hospital admission | stay duration and patient sequence |
| `employees` | one row per employee | hierarchy and salary ranking |
| `sales` | one row per sale | running totals, moving averages, top products |
| `website_visits` | one row per website visit | next-event analysis |

**Grain** means: *What does one row represent?*

## 5. Create or reset the teaching tables

Run this cell once at the beginning. Re-run it whenever you want to restore the original Module 11 sample data.

In [ ]:
setup_sql = """
DROP TABLE IF EXISTS website_visits CASCADE;
DROP TABLE IF EXISTS sales CASCADE;
DROP TABLE IF EXISTS employees CASCADE;
DROP TABLE IF EXISTS admissions CASCADE;
DROP TABLE IF EXISTS patients CASCADE;

CREATE TABLE patients (
    subject_id integer PRIMARY KEY,
    name text NOT NULL,
    age integer NOT NULL CHECK (age >= 0)
);

CREATE TABLE admissions (
    hadm_id integer PRIMARY KEY,
    subject_id integer NOT NULL REFERENCES patients(subject_id),
    admit_time timestamp NOT NULL,
    discharge_time timestamp,
    department text NOT NULL,
    diagnosis text NOT NULL
);

CREATE TABLE employees (
    employee_id integer PRIMARY KEY,
    employee_name text NOT NULL,
    department text NOT NULL,
    salary numeric(10,2) NOT NULL CHECK (salary > 0),
    manager_id integer REFERENCES employees(employee_id)
);

CREATE TABLE sales (
    sale_id integer PRIMARY KEY,
    sale_date date NOT NULL,
    product text NOT NULL,
    department text NOT NULL,
    revenue numeric(10,2) NOT NULL CHECK (revenue >= 0)
);

CREATE TABLE website_visits (
    visit_id integer PRIMARY KEY,
    user_id integer NOT NULL,
    visit_timestamp timestamp NOT NULL
);

INSERT INTO patients VALUES
(1001,'Alice Smith',45),
(1002,'Bob Johnson',32),
(1003,'Carol White',58),
(1004,'David Brown',67),
(1005,'Eve Davis',29),
(1006,'Grace Wilson',52);

INSERT INTO admissions VALUES
(101,1001,'2025-01-10 08:00','2025-01-15 12:00','Cardiology','Pneumonia'),
(102,1002,'2025-02-01 14:00','2025-02-03 09:00','Orthopedics','Fracture'),
(103,1001,'2025-03-05 10:00','2025-03-12 15:00','Internal Medicine','Hypertension'),
(104,1003,'2025-04-10 07:00','2025-04-11 11:00','Neurology','Migraine'),
(105,1001,'2025-05-15 09:00','2025-05-20 14:00','Cardiology','Chest pain'),
(106,1002,'2025-02-20 09:30','2025-02-22 10:00','Orthopedics','Follow-up'),
(107,1006,'2025-05-01 07:30','2025-05-04 11:30','Internal Medicine','Asthma'),
(108,1006,'2025-05-25 10:00','2025-05-28 13:00','Internal Medicine','Asthma');

INSERT INTO employees VALUES
(1,'Nora Patel','Executive',150000,NULL),
(2,'Amir Chen','Analytics',125000,1),
(3,'Beth Morgan','Analytics',125000,1),
(4,'Chen Wu','Analytics',95000,2),
(5,'Dana Ruiz','Analytics',95000,2),
(6,'Eli Brooks','Engineering',115000,1),
(7,'Fatima Noor','Engineering',105000,6),
(8,'Gio Rossi','Engineering',105000,6),
(9,'Hana Kim','IT',85000,1);

INSERT INTO sales VALUES
(1,'2025-01-01','Laptop','Electronics',1000),
(2,'2025-01-01','Monitor','Electronics',1000),
(3,'2025-01-02','Laptop','Electronics',1200),
(4,'2025-01-02','Mouse','Accessories',200),
(5,'2025-01-03','Keyboard','Accessories',300),
(6,'2025-01-03','Mouse','Accessories',300),
(7,'2025-01-04','Laptop','Electronics',1500),
(8,'2025-01-04','Monitor','Electronics',1300),
(9,'2025-01-05','Mouse','Accessories',400),
(10,'2025-01-05','Keyboard','Accessories',400);

INSERT INTO website_visits VALUES
(1,501,'2025-06-01 08:00'),
(2,501,'2025-06-01 10:30'),
(3,501,'2025-06-02 09:15'),
(4,502,'2025-06-01 11:00'),
(5,502,'2025-06-03 14:00'),
(6,503,'2025-06-02 16:20');
"""

execute_script(setup_sql)

## 6. Confirm the tables were loaded

In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM patients) AS patients,
    (SELECT COUNT(*) FROM admissions) AS admissions,
    (SELECT COUNT(*) FROM employees) AS employees,
    (SELECT COUNT(*) FROM sales) AS sales,
    (SELECT COUNT(*) FROM website_visits) AS website_visits;
""", title="Row counts")

## 7. Inspect the base tables

In [ ]:
display_query("SELECT * FROM patients ORDER BY subject_id;", title="Patients")

In [ ]:
display_query("SELECT * FROM admissions ORDER BY subject_id, admit_time;", title="Admissions")

In [ ]:
display_query("SELECT * FROM employees ORDER BY employee_id;", title="Employees")

In [ ]:
display_query("SELECT * FROM sales ORDER BY sale_date, sale_id;", title="Sales")

In [ ]:
display_query("SELECT * FROM website_visits ORDER BY user_id, visit_timestamp;", title="Website visits")

# Part II · Common Table Expressions (CTEs)

A **CTE** gives an intermediate query result a name.

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT ...
FROM cte_name;
```

Use CTEs when a logical stage deserves a name. They are especially useful for readable analytical pipelines, ranking-then-filtering, and recursive queries.

A CTE is not automatically a permanent table, a cache, or a performance improvement.

<!-- STEP_TABLES_V2 -->
## Start with the idea before the syntax

A CTE is easiest to understand as a **temporary named result inside one query**.

Suppose the original `patients` table contains:

| subject_id | name | age |
|---:|---|---:|
| 1001 | Alice Smith | 45 |
| 1003 | Carol White | 58 |
| 1004 | David Brown | 67 |
| 1006 | Grace Wilson | 52 |

If we write:

```sql
WITH older_patients AS (
    SELECT subject_id, name, age
    FROM patients
    WHERE age > 50
)
```

the CTE temporarily behaves like this smaller table:

| subject_id | name | age |
|---:|---|---:|
| 1003 | Carol White | 58 |
| 1004 | David Brown | 67 |
| 1006 | Grace Wilson | 52 |

Then the rest of the SQL can query `older_patients` as if it were a table.

### Mental model

```text
patients
   ↓ filter
older_patients
   ↓ join / aggregate / calculate
final result
```

The important idea is not the word `WITH`. The important idea is that we **name one stage of the problem**, then continue from that stage.

## 8. Filter first, then analyze

`older_patients` is at **patient grain**.  
The final result is also one row per older patient.

### Step 1 — Build the CTE result

First, ignore the final query. Ask only:

> Which patients belong in `older_patients`?

Expected intermediate result:

| subject_id | name | age |
|---:|---|---:|
| 1003 | Carol White | 58 |
| 1004 | David Brown | 67 |
| 1006 | Grace Wilson | 52 |

Run the intermediate query by itself first.

In [ ]:
display_query("""
SELECT
    subject_id,
    name,
    age
FROM patients
WHERE age > 50
ORDER BY subject_id;
""", title="Step 1: Rows that will become older_patients")

### Step 2 — Join those rows to admissions

Conceptually, the join expands each qualifying patient to any matching admission rows.

A simplified view looks like this:

| subject_id | patient | hadm_id |
|---:|---|---:|
| 1003 | Carol White | 104 |
| 1004 | David Brown | NULL |
| 1006 | Grace Wilson | 107 |
| 1006 | Grace Wilson | 108 |

Notice why a `LEFT JOIN` matters: David Brown remains in the result even though there is no matching admission.

### Step 3 — Aggregate back to one row per patient

After `COUNT(a.hadm_id)` and `GROUP BY`, the grain becomes:

> **one row per older patient**

| subject_id | patient | age | admission_count |
|---:|---|---:|---:|
| 1004 | David Brown | 67 | 0 |
| 1003 | Carol White | 58 | 1 |
| 1006 | Grace Wilson | 52 | 2 |

Now the full CTE query below should be easier to read.

In [ ]:
display_query("""
WITH older_patients AS (
    SELECT subject_id, name, age
    FROM patients
    WHERE age > 50
)
SELECT
    op.subject_id,
    op.name,
    op.age,
    COUNT(a.hadm_id) AS admission_count
FROM older_patients AS op
LEFT JOIN admissions AS a
    ON a.subject_id = op.subject_id
GROUP BY op.subject_id, op.name, op.age
ORDER BY op.age DESC;
""", title="Older patients CTE")

### Follow the stages

1. Keep patients older than 50.
2. Join admissions.
3. Group by patient.
4. Return one row per older patient.

## 9. Several CTEs can form a dependency chain

### Think of chained CTEs as a pipeline

The query in this section has three stages:

| Stage | Input | Output grain | Purpose |
|---|---|---|---|
| `admission_counts` | `admissions` | one row per patient with admissions | count admissions |
| `patient_features` | `patients` + counts | one row per patient | attach the count to every patient |
| final `SELECT` | `patient_features` | one row per patient | classify visit pattern |

A small example:

**Stage 1 — `admission_counts`**

| subject_id | admission_count |
|---:|---:|
| 1001 | 3 |
| 1002 | 2 |
| 1003 | 1 |
| 1006 | 2 |

**Stage 2 — after joining to all patients**

| subject_id | patient | admission_count |
|---:|---|---:|
| 1001 | Alice Smith | 3 |
| 1002 | Bob Johnson | 2 |
| 1003 | Carol White | 1 |
| 1004 | David Brown | 0 |
| 1005 | Eve Davis | 0 |
| 1006 | Grace Wilson | 2 |

**Stage 3 — classify**

| admission_count | visit_pattern |
|---:|---|
| 0 or 1 | `single_or_none` |
| 2 or more | `repeat` |

This is a useful habit: make **each CTE responsible for one clear transformation**.

In [ ]:
display_query("""
SELECT
    subject_id,
    COUNT(*) AS admission_count
FROM admissions
GROUP BY subject_id
ORDER BY subject_id;
""", title="Step 1: admission_counts by itself")

In [ ]:
display_query("""
WITH admission_counts AS (
    SELECT
        subject_id,
        COUNT(*) AS admission_count
    FROM admissions
    GROUP BY subject_id
),
patient_features AS (
    SELECT
        p.subject_id,
        p.name,
        p.age,
        COALESCE(a.admission_count, 0) AS admission_count
    FROM patients AS p
    LEFT JOIN admission_counts AS a
        USING (subject_id)
)
SELECT
    *,
    CASE
        WHEN admission_count >= 2 THEN 'repeat'
        ELSE 'single_or_none'
    END AS visit_pattern
FROM patient_features
ORDER BY admission_count DESC, subject_id;
""", title="Chained CTEs")

## 10. Prepare a reusable measure

The CTE computes stay duration at admission grain.  
The outer query summarizes it at patient grain.

### Why calculate `stay_days` first?

The raw table stores timestamps:

| hadm_id | admit_time | discharge_time |
|---:|---|---|
| 101 | Jan 10 08:00 | Jan 15 12:00 |
| 103 | Mar 05 10:00 | Mar 12 15:00 |

But the analysis needs a numeric measure: **length of stay in days**.

So the CTE converts each admission into a cleaner analytical row:

| hadm_id | subject_id | stay_days |
|---:|---:|---:|
| 101 | 1001 | 5.17 |
| 103 | 1001 | 7.21 |
| 105 | 1001 | 5.21 |

Only after that do we compute the patient average.

This separation makes debugging easier:
1. verify the row-level measure,
2. then verify the summary.

In [ ]:
display_query("""
SELECT
    hadm_id,
    subject_id,
    ROUND(
        (EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0)::numeric,
        2
    ) AS stay_days
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY subject_id, admit_time;
""", title="Step 1: Verify stay_days before averaging")

In [ ]:
display_query("""
WITH stay_days AS (
    SELECT
        hadm_id,
        subject_id,
        department,
        EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0 AS stay_days
    FROM admissions
    WHERE discharge_time IS NOT NULL
)
SELECT
    p.name,
    COUNT(s.hadm_id) AS admissions,
    ROUND(AVG(s.stay_days), 2) AS avg_stay_days
FROM patients AS p
LEFT JOIN stay_days AS s
    USING (subject_id)
GROUP BY p.subject_id, p.name
ORDER BY admissions DESC, p.subject_id;
""", title="Prepare stay days, then summarize")

## 11. Use `EXPLAIN` instead of guessing about performance

PostgreSQL may optimize a non-recursive CTE together with the surrounding query. Use the plan when performance matters.

In [ ]:
display_query("""
EXPLAIN
WITH older_patients AS (
    SELECT *
    FROM patients
    WHERE age > 50
)
SELECT *
FROM older_patients
WHERE subject_id > 1002;
""", title="EXPLAIN a CTE query")

# Part III · Recursive CTEs

A recursive CTE repeatedly feeds results into the next step.

It contains:
- an **anchor** query,
- a **recursive term**,
- usually `UNION ALL`,
- and a stopping condition created naturally when no more rows match.

<!-- RECURSION_STEP_TABLES_V2 -->
## Recursion: think in levels, not all at once

A recursive CTE can look intimidating because one CTE refers to itself. The easiest way to understand it is to manually trace a tiny hierarchy.

Suppose we have:

| employee_id | employee | manager_id |
|---:|---|---:|
| 1 | Nora | NULL |
| 2 | Amir | 1 |
| 3 | Beth | 1 |
| 4 | Chen | 2 |

This means:

```text
Nora
├── Amir
│   └── Chen
└── Beth
```

### Iteration 0 — Anchor

The anchor finds the starting row:

| employee | depth |
|---|---:|
| Nora | 0 |

### Iteration 1 — Find children of Nora

| employee | depth |
|---|---:|
| Amir | 1 |
| Beth | 1 |

### Iteration 2 — Find children of the previous level

| employee | depth |
|---|---:|
| Chen | 2 |

### Iteration 3

No more matching children are found, so recursion stops.

The recursive CTE is therefore building the result **level by level**.

## 12. Traverse an employee hierarchy

### Step 1 — Find only the root

Before recursion, verify the anchor query:

In [ ]:
display_query("""
SELECT
    employee_id,
    employee_name,
    manager_id,
    0 AS depth
FROM employees
WHERE manager_id IS NULL;
""", title="Recursive CTE Step 1: Anchor row")

### Step 2 — Find the root's direct reports

If the root employee has `employee_id = 1`, the next level is simply:

```sql
WHERE manager_id = 1
```

That gives a concrete picture of what the recursive term will do automatically on each iteration.

In [ ]:
display_query("""
SELECT
    employee_id,
    employee_name,
    manager_id
FROM employees
WHERE manager_id = 1
ORDER BY employee_id;
""", title="Recursive CTE Step 2: One manual expansion")

### Step 3 — Let SQL repeat the expansion

The recursive part:

```sql
JOIN org AS o
    ON e.manager_id = o.employee_id
```

means:

> For every employee already found in `org`, find employees whose `manager_id` points to that employee.

The `depth` column tells us how many edges away from the root we are, while `path` helps us see how each employee was reached.

In [ ]:
display_query("""
WITH RECURSIVE org AS (

    SELECT
        employee_id,
        employee_name,
        manager_id,
        0 AS depth,
        employee_name::text AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT
        e.employee_id,
        e.employee_name,
        e.manager_id,
        o.depth + 1,
        o.path || ' > ' || e.employee_name
    FROM employees AS e
    JOIN org AS o
        ON e.manager_id = o.employee_id
)

SELECT
    employee_id,
    employee_name,
    manager_id,
    depth,
    path
FROM org
ORDER BY path;
""", title="Recursive organization traversal")

### Read it one level at a time

- Depth 0: the anchor finds the root employee.
- Depth 1: find the root's direct reports.
- Depth 2: find their reports.
- Continue until no more matching employees are found.

When testing recursion, inspect `depth` and `path`, and be careful about cycles.

# Part IV · Window Functions

A window function calculates across related rows **without automatically collapsing the current row**.

| `GROUP BY` | Window function |
|---|---|
| reduces rows | preserves detail rows |
| changes output grain | adds context beside detail |
| use for group-level output | use when detail and group context must coexist |

General pattern:

```sql
function(...) OVER (
    PARTITION BY ...
    ORDER BY ...
    ROWS BETWEEN ...
)
```

<!-- WINDOW_STEP_TABLES_V2 -->
## Window functions: keep the row, add context

Consider this tiny salary table:

| employee | department | salary |
|---|---|---:|
| Amir | Analytics | 125,000 |
| Beth | Analytics | 125,000 |
| Chen | Analytics | 95,000 |
| Dana | Analytics | 95,000 |
| Eli | Engineering | 115,000 |
| Fatima | Engineering | 105,000 |
| Gio | Engineering | 105,000 |

A normal `GROUP BY department` could return:

| department | avg_salary |
|---|---:|
| Analytics | 110,000 |
| Engineering | 108,333.33 |

But the individual employee rows disappear.

A window calculation keeps them:

| employee | department | salary | department_avg |
|---|---|---:|---:|
| Amir | Analytics | 125,000 | 110,000 |
| Beth | Analytics | 125,000 | 110,000 |
| Chen | Analytics | 95,000 | 110,000 |
| Dana | Analytics | 95,000 | 110,000 |
| Eli | Engineering | 115,000 | 108,333.33 |
| Fatima | Engineering | 105,000 | 108,333.33 |
| Gio | Engineering | 105,000 | 108,333.33 |

That is the central idea of a window function:

> **Keep the detail row, but let the row see related rows.**

## 13. Four pieces of a window calculation

1. **Function** — what is calculated?
2. **PARTITION** — where does it restart?
3. **ORDER** — what is the sequence?
4. **Frame** — which ordered rows contribute?

### A useful reading order

When you see:

```sql
AVG(salary) OVER (
    PARTITION BY department
)
```

read it as:

1. **`AVG(salary)`** → what calculation?
2. **`OVER(...)`** → calculate it as a window.
3. **`PARTITION BY department`** → restart separately for each department.

For:

```sql
ROW_NUMBER() OVER (
    PARTITION BY department
    ORDER BY salary DESC
)
```

add one more question:

4. **`ORDER BY salary DESC`** → in what sequence should the department rows be numbered?

## 14. Department average while keeping employee rows

### Step-by-step partition view

Imagine SQL temporarily separates the rows into baskets.

**Analytics partition**

| employee | salary |
|---|---:|
| Amir | 125,000 |
| Beth | 125,000 |
| Chen | 95,000 |
| Dana | 95,000 |

**Engineering partition**

| employee | salary |
|---|---:|
| Eli | 115,000 |
| Fatima | 105,000 |
| Gio | 105,000 |

The average is calculated **inside each basket**, then copied beside every row in that basket.

`ROW_NUMBER()` uses the same partitions but also needs an order.

In [ ]:
display_query("""
SELECT
    employee_id,
    employee_name,
    department,
    salary,
    AVG(salary) OVER (
        PARTITION BY department
    ) AS department_avg_salary,
    ROW_NUMBER() OVER (
        PARTITION BY department
        ORDER BY salary DESC, employee_id
    ) AS row_num
FROM employees
ORDER BY department, row_num;
""", title="Average salary and row number")

`PARTITION BY department` creates independent department calculations.

`ORDER BY salary DESC, employee_id` defines the analytical order used by `ROW_NUMBER()`.

# Part V · Ranking Functions

| Function | Tie behavior | Typical use |
|---|---|---|
| `ROW_NUMBER()` | always unique | exactly N rows |
| `RANK()` | ties share rank; gaps follow | competition-style ranking |
| `DENSE_RANK()` | ties share rank; no gaps | distinct rank groups |

If values tie:

- `ROW_NUMBER()` → `1, 2, 3, 4`
- `RANK()` → `1, 2, 2, 4`
- `DENSE_RANK()` → `1, 2, 2, 3`

<!-- RANKING_STEP_TABLES_V2 -->
## Why three ranking functions exist

Use this tiny ordered salary list:

| employee | salary |
|---|---:|
| A | 120 |
| B | 100 |
| C | 100 |
| D | 80 |

Now compare:

| employee | salary | ROW_NUMBER | RANK | DENSE_RANK |
|---|---:|---:|---:|---:|
| A | 120 | 1 | 1 | 1 |
| B | 100 | 2 | 2 | 2 |
| C | 100 | 3 | 2 | 2 |
| D | 80 | 4 | 4 | 3 |

### Choose based on the question

- **ROW_NUMBER**: “Give me exactly two rows.”
- **RANK**: “Give equal values the same competition rank.”
- **DENSE_RANK**: “Give equal values the same rank, but do not leave gaps.”

The syntax is similar, but the business meaning is different.

## 15. Compare the ranking functions

### Predict before running

The Module 11 employee table intentionally includes salary ties.

Before executing the query, predict:
- Which employees should share a rank?
- Where should `RANK()` create gaps?
- Why will `ROW_NUMBER()` still assign unique numbers?

Prediction is useful here because ranking mistakes often produce valid SQL but the wrong business answer.

In [ ]:
display_query("""
SELECT
    employee_name,
    department,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC, employee_id
    ) AS row_number,
    RANK() OVER (
        ORDER BY salary DESC
    ) AS rank,
    DENSE_RANK() OVER (
        ORDER BY salary DESC
    ) AS dense_rank
FROM employees
ORDER BY salary DESC, employee_id;
""", title="ROW_NUMBER vs RANK vs DENSE_RANK")

### Tie-breakers

When a unique sequence is required, add a stable tie-breaker:

```sql
ORDER BY salary DESC, employee_id
```

For `RANK()` or `DENSE_RANK()`, do not add a tie-breaker if equal salaries are intentionally supposed to remain tied.

## 16. Independent rankings inside each department

In [ ]:
display_query("""
SELECT
    employee_name,
    department,
    salary,
    RANK() OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS salary_rank
FROM employees
ORDER BY department, salary_rank, employee_id;
""", title="Department salary ranks")

# Part VI · Sequence Analysis with LAG and LEAD

`LAG()` retrieves an earlier ordered value.  
`LEAD()` retrieves a later ordered value.

Common uses:
- previous admission,
- previous discharge,
- next website visit,
- change from previous measurement,
- time until next event.

Previous and next only make sense when the order is meaningful.

<!-- LAG_LEAD_STEP_TABLES_V2 -->
## Think of `LAG()` and `LEAD()` as shifting a column

Suppose one patient's admissions are ordered like this:

| row | admit_time | discharge_time |
|---:|---|---|
| 1 | Jan 10 | Jan 15 |
| 2 | Mar 05 | Mar 12 |
| 3 | May 15 | May 20 |

`LAG(discharge_time)` shifts the previous discharge beside the current row:

| current admit | current discharge | previous discharge |
|---|---|---|
| Jan 10 | Jan 15 | NULL |
| Mar 05 | Mar 12 | Jan 15 |
| May 15 | May 20 | Mar 12 |

Now subtraction becomes easy:

```text
current admit − previous discharge = gap
```

The first row gets `NULL` because there is no earlier row in that patient's partition.

## 17. Previous discharge with `LAG()`

### Step 1 — Define independent sequences

We use:

```sql
PARTITION BY subject_id
```

so each patient gets a separate timeline.

### Step 2 — Define the order

We use:

```sql
ORDER BY admit_time, hadm_id
```

so “previous” has a precise meaning.

### Step 3 — Shift the value

`LAG(discharge_time)` brings the previous row's discharge time beside the current admission.

### Step 4 — Calculate the interval

Only after the previous value exists do we subtract timestamps.

In [ ]:
display_query("""
WITH sequenced AS (
    SELECT
        hadm_id,
        subject_id,
        admit_time,
        discharge_time,
        LAG(discharge_time) OVER (
            PARTITION BY subject_id
            ORDER BY admit_time, hadm_id
        ) AS previous_discharge
    FROM admissions
)
SELECT
    hadm_id,
    subject_id,
    admit_time,
    previous_discharge,
    admit_time - previous_discharge AS time_since_discharge
FROM sequenced
WHERE previous_discharge IS NOT NULL
ORDER BY subject_id, admit_time;
""", title="Previous discharge with LAG")

## 18. Next website visit with `LEAD()`

### Mini example for `LEAD()`

For one website user:

| visit | timestamp |
|---:|---|
| 1 | Jun 1 08:00 |
| 2 | Jun 1 10:30 |
| 3 | Jun 2 09:15 |

After `LEAD(visit_timestamp)`:

| current visit | next visit |
|---|---|
| Jun 1 08:00 | Jun 1 10:30 |
| Jun 1 10:30 | Jun 2 09:15 |
| Jun 2 09:15 | NULL |

`LEAD()` looks forward; `LAG()` looks backward.

In [ ]:
display_query("""
SELECT
    user_id,
    visit_timestamp,
    LEAD(visit_timestamp) OVER (
        PARTITION BY user_id
        ORDER BY visit_timestamp, visit_id
    ) AS next_visit,
    LEAD(visit_timestamp) OVER (
        PARTITION BY user_id
        ORDER BY visit_timestamp, visit_id
    ) - visit_timestamp AS time_to_next_visit
FROM website_visits
ORDER BY user_id, visit_timestamp;
""", title="Next visit with LEAD")

For event sequences, a timestamp often gives the main order and an ID gives a stable tie-breaker:

```sql
ORDER BY event_timestamp, event_id
```

# Part VII · Window Frames

Important boundaries:

- `UNBOUNDED PRECEDING` — start at the beginning
- `CURRENT ROW` — current ordered row
- `2 PRECEDING` — include two earlier physical rows
- `UNBOUNDED FOLLOWING` — extend through the end

<!-- FRAME_STEP_TABLES_V2 -->
## A frame is the set of rows used for the current calculation

Assume daily revenue is:

| row | day | revenue |
|---:|---|---:|
| 1 | Day 1 | 100 |
| 2 | Day 2 | 200 |
| 3 | Day 3 | 300 |
| 4 | Day 4 | 400 |

### Running total frame

For Day 3:

```text
[Day 1, Day 2, Day 3]
```

Result: `100 + 200 + 300 = 600`

### 3-row moving frame

For Day 4:

```text
[Day 2, Day 3, Day 4]
```

Result for a moving average: `(200 + 300 + 400) / 3 = 300`

The current row stays the same; the **frame changes which neighboring rows participate**.

## 19. Running revenue by department

### Two operations happen here

**First:** reduce raw sales to daily department totals.

Example:

| sale_date | department | daily_revenue |
|---|---|---:|
| Jan 2 | Electronics | 1,200 |
| Jan 4 | Electronics | 2,800 |

**Second:** calculate a running total over those daily rows.

| sale_date | daily_revenue | running_revenue |
|---|---:|---:|
| Jan 2 | 1,200 | 1,200 |
| Jan 4 | 2,800 | 4,000 |

This is why the CTE comes first: we want the window to operate on the **daily grain**, not the raw transaction grain.

In [ ]:
display_query("""
WITH daily AS (
    SELECT
        sale_date,
        department,
        SUM(revenue) AS daily_revenue
    FROM sales
    GROUP BY sale_date, department
)
SELECT
    sale_date,
    department,
    daily_revenue,
    SUM(daily_revenue) OVER (
        PARTITION BY department
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue
FROM daily
ORDER BY department, sale_date;
""", title="Running department revenue")

The frame:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

means: start at the beginning of the partition and accumulate through the current row.

## 20. Three-row moving average

### Trace one row manually

If daily revenue is:

| day | revenue |
|---|---:|
| Day 1 | 2,000 |
| Day 2 | 1,400 |
| Day 3 | 600 |

For Day 3, this frame:

```sql
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
```

means the average uses all three rows:

```text
(2000 + 1400 + 600) / 3
```

For the very first row, only one row is available. PostgreSQL does not invent missing earlier rows.

In [ ]:
display_query("""
WITH daily AS (
    SELECT
        sale_date,
        SUM(revenue) AS daily_revenue
    FROM sales
    GROUP BY sale_date
)
SELECT
    sale_date,
    daily_revenue,
    ROUND(
        AVG(daily_revenue) OVER (
            ORDER BY sale_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS moving_avg_3_rows
FROM daily
ORDER BY sale_date;
""", title="Three-row moving average")

`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` uses the current row plus up to two earlier rows.

This is a **three-row** window. It is not automatically a three-calendar-day window.

## 21. Explicit cumulative frame at sale grain

In [ ]:
display_query("""
SELECT
    sale_id,
    sale_date,
    product,
    revenue,
    SUM(revenue) OVER (
        ORDER BY sale_date, sale_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue
FROM sales
ORDER BY sale_date, sale_id;
""", title="Explicit cumulative frame")

## 22. `LAST_VALUE()` and frame behavior

`LAST_VALUE()` uses the current window frame. It does not automatically mean “the final row in the entire partition.”

Compare the default ordered frame with a full-partition frame.

### Why the default can surprise you

Suppose values are:

| row | revenue |
|---:|---:|
| 1 | 100 |
| 2 | 200 |
| 3 | 300 |

With an ordered default frame, row 2 may only see through row 2. Therefore `LAST_VALUE(revenue)` can return `200`, not `300`.

To explicitly ask for the last value of the **whole partition**, extend the frame through:

```sql
UNBOUNDED FOLLOWING
```

This is an important lesson: the function name alone does not define the result; the **frame** also matters.

In [ ]:
display_query("""
SELECT
    sale_id,
    sale_date,
    revenue,
    LAST_VALUE(revenue) OVER (
        ORDER BY sale_date, sale_id
    ) AS default_frame_last_value,
    LAST_VALUE(revenue) OVER (
        ORDER BY sale_date, sale_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS partition_last_value
FROM sales
ORDER BY sale_date, sale_id;
""", title="LAST_VALUE frame behavior")

# Part VIII · GROUP BY vs PARTITION BY

Use `GROUP BY` when the group-level result is the output you want.

Use a window with `PARTITION BY` when you want the detail rows to remain visible while adding group context.

<!-- GROUP_WINDOW_STEP_TABLES_V2 -->
## Same input, different output shape

Start with:

| hadm_id | subject_id | stay_days |
|---:|---:|---:|
| 101 | 1001 | 5 |
| 103 | 1001 | 7 |
| 105 | 1001 | 5 |
| 102 | 1002 | 2 |

### With `GROUP BY subject_id`

Rows collapse:

| subject_id | avg_stay |
|---:|---:|
| 1001 | 5.67 |
| 1002 | 2.00 |

The admission IDs disappear because the output is now **patient grain**.

### With `AVG(...) OVER (PARTITION BY subject_id)`

Rows remain:

| hadm_id | subject_id | stay_days | patient_avg |
|---:|---:|---:|---:|
| 101 | 1001 | 5 | 5.67 |
| 103 | 1001 | 7 | 5.67 |
| 105 | 1001 | 5 | 5.67 |
| 102 | 1002 | 2 | 2.00 |

### One question usually decides which one you need

> Do I want **one row per group**, or do I want to **keep the detail rows**?

That question is more useful than memorizing syntax.

## 23. Patient average stay with `GROUP BY`

In [ ]:
display_query("""
SELECT
    subject_id,
    ROUND(
        AVG(
            EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0
        ),
        2
    ) AS avg_stay_days
FROM admissions
WHERE discharge_time IS NOT NULL
GROUP BY subject_id
ORDER BY subject_id;
""", title="Average stay with GROUP BY")

## 24. Patient average stay with `PARTITION BY`

In [ ]:
display_query("""
SELECT
    hadm_id,
    subject_id,
    admit_time,
    ROUND(
        EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0,
        2
    ) AS stay_days,
    ROUND(
        AVG(
            EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0
        ) OVER (
            PARTITION BY subject_id
        ),
        2
    ) AS patient_avg_stay_days
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY subject_id, admit_time;
""", title="Average stay with PARTITION BY")

## 25. Compare output row counts

In [ ]:
display_query("""
SELECT
    'GROUP BY' AS method,
    COUNT(*) AS output_rows
FROM (
    SELECT
        subject_id,
        AVG(discharge_time - admit_time)
    FROM admissions
    GROUP BY subject_id
) AS g

UNION ALL

SELECT
    'PARTITION BY',
    COUNT(*)
FROM (
    SELECT
        hadm_id,
        AVG(discharge_time - admit_time) OVER (
            PARTITION BY subject_id
        )
    FROM admissions
) AS w;
""", title="GROUP BY vs PARTITION BY row counts")

# Part IX · Advanced Analytical Patterns

## 26. Percent of total while preserving every sale

`SUM(revenue) OVER ()` calculates the overall total while keeping each sale row.

In [ ]:
percent_df = display_query("""
SELECT
    sale_id,
    product,
    revenue,
    ROUND(
        100.0 * revenue / SUM(revenue) OVER (),
        2
    ) AS percent_of_all_revenue
FROM sales
ORDER BY revenue DESC, sale_id;
""", title="Percent of all revenue", return_df=True)

if not percent_df.empty:
    total_pct = percent_df["percent_of_all_revenue"].astype(float).sum()
    print("Displayed percentage total:", round(total_pct, 2))

## 27. Rank first, filter second

A common Top-N-per-group pattern is:

1. calculate a ranking in a CTE,
2. filter that ranking in the outer query.

### Why filtering needs another stage

Suppose department salaries are ranked:

| employee | department | salary | rn |
|---|---|---:|---:|
| Amir | Analytics | 125,000 | 1 |
| Beth | Analytics | 125,000 | 2 |
| Chen | Analytics | 95,000 | 3 |
| Dana | Analytics | 95,000 | 4 |

Then the outer query can simply apply:

```sql
WHERE rn <= 2
```

and keep:

| employee | department | salary | rn |
|---|---|---:|---:|
| Amir | Analytics | 125,000 | 1 |
| Beth | Analytics | 125,000 | 2 |

The pattern is:

```text
raw rows
   ↓
calculate ranking
   ↓
filter ranking
```

This pattern appears constantly in analytics SQL.

### Exactly two employees per department with `ROW_NUMBER()`

In [ ]:
display_query("""
WITH ranked AS (
    SELECT
        employee_id,
        employee_name,
        department,
        salary,
        ROW_NUMBER() OVER (
            PARTITION BY department
            ORDER BY salary DESC, employee_id
        ) AS rn
    FROM employees
)
SELECT
    employee_id,
    employee_name,
    department,
    salary,
    rn
FROM ranked
WHERE rn <= 2
ORDER BY department, rn;
""", title="Top two salaries per department")

### Include tied rank groups with `RANK()`

In [ ]:
display_query("""
WITH ranked AS (
    SELECT
        employee_id,
        employee_name,
        department,
        salary,
        RANK() OVER (
            PARTITION BY department
            ORDER BY salary DESC
        ) AS salary_rank
    FROM employees
)
SELECT
    employee_id,
    employee_name,
    department,
    salary,
    salary_rank
FROM ranked
WHERE salary_rank <= 2
ORDER BY department, salary_rank, employee_id;
""", title="Top salary ranks including ties")

## 28. Aggregate → rank → filter

First create the correct grain, then rank it.

Here the first CTE creates one row per `department + product`.

### Grain changes twice in this pattern

1. Raw `sales` = **one row per sale**
2. `product_totals` = **one row per department + product**
3. `ranked` = same product grain, plus a rank
4. final result = only the desired rank groups

A small conceptual example:

| department | product | product_revenue |
|---|---|---:|
| Electronics | Laptop | 3,700 |
| Electronics | Monitor | 2,300 |
| Accessories | Mouse | 900 |
| Accessories | Keyboard | 700 |

After ranking inside each department:

| department | product | product_revenue | rank |
|---|---|---:|---:|
| Electronics | Laptop | 3,700 | 1 |
| Electronics | Monitor | 2,300 | 2 |
| Accessories | Mouse | 900 | 1 |
| Accessories | Keyboard | 700 | 2 |

Always create the grain you want to rank **before** applying the ranking function.

In [ ]:
display_query("""
WITH product_totals AS (
    SELECT
        department,
        product,
        SUM(revenue) AS product_revenue
    FROM sales
    GROUP BY department, product
),
ranked AS (
    SELECT
        department,
        product,
        product_revenue,
        DENSE_RANK() OVER (
            PARTITION BY department
            ORDER BY product_revenue DESC
        ) AS revenue_rank
    FROM product_totals
)
SELECT
    department,
    product,
    product_revenue,
    revenue_rank
FROM ranked
WHERE revenue_rank <= 2
ORDER BY department, revenue_rank, product;
""", title="Top products within department")

# Part X · Applied Example: Readmission Interval Analysis

Pipeline:

1. sequence admissions within each patient,
2. retrieve previous discharge with `LAG()`,
3. calculate the gap,
4. classify whether the gap is within 30 days.

| Stage | Grain | Purpose |
|---|---|---|
| `sequenced` | one admission | add previous discharge |
| `intervals` | one repeat admission | calculate gap |
| final `SELECT` | one repeat admission | add patient and classification |

<!-- READMISSION_STEP_TABLES_V2 -->
## Walk through the analysis one transformation at a time

Consider one patient:

| hadm_id | admit | discharge |
|---:|---|---|
| 101 | Jan 10 | Jan 15 |
| 103 | Mar 05 | Mar 12 |
| 105 | May 15 | May 20 |

### Stage 1 — Sequence admissions and add previous discharge

| hadm_id | admit | previous_discharge |
|---:|---|---|
| 101 | Jan 10 | NULL |
| 103 | Mar 05 | Jan 15 |
| 105 | May 15 | Mar 12 |

### Stage 2 — Remove rows with no previous admission and calculate the gap

| hadm_id | admit | previous_discharge | gap |
|---:|---|---|---|
| 103 | Mar 05 | Jan 15 | about 49 days |
| 105 | May 15 | Mar 12 | about 64 days |

### Stage 3 — Classify

| gap | readmission_flag |
|---|---|
| `<= 30 days` | `within_30_days` |
| `> 30 days` | `over_30_days` |

Notice how each stage does only one job. That makes a complex clinical-style analysis much easier to test.

In [ ]:
display_query("""
WITH sequenced AS (
    SELECT
        a.*,
        LAG(discharge_time) OVER (
            PARTITION BY subject_id
            ORDER BY admit_time, hadm_id
        ) AS previous_discharge
    FROM admissions AS a
),
intervals AS (
    SELECT
        hadm_id,
        subject_id,
        admit_time,
        previous_discharge,
        admit_time - previous_discharge AS gap
    FROM sequenced
    WHERE previous_discharge IS NOT NULL
)
SELECT
    p.name,
    i.hadm_id,
    i.admit_time,
    i.previous_discharge,
    i.gap,
    CASE
        WHEN i.gap <= INTERVAL '30 days'
            THEN 'within_30_days'
        ELSE 'over_30_days'
    END AS readmission_flag
FROM intervals AS i
JOIN patients AS p
    USING (subject_id)
ORDER BY p.name, i.admit_time;
""", title="Readmission interval analysis")

# Part XI · Validation Habits

Do not stop at “the SQL ran.”

Also ask:
- Did the row count change?
- Should it have changed?
- What is the output grain?
- Are ties handled correctly?
- Do running totals reconcile?
- Are `NULL`s from `LAG()`/`LEAD()` expected?

## 29. Compare grouped and window output grain

In [ ]:
display_query("""
SELECT
    (SELECT COUNT(*) FROM admissions) AS source_admission_rows,
    (
        SELECT COUNT(*)
        FROM (
            SELECT subject_id, COUNT(*)
            FROM admissions
            GROUP BY subject_id
        ) AS g
    ) AS grouped_patient_rows,
    (
        SELECT COUNT(*)
        FROM (
            SELECT
                hadm_id,
                subject_id,
                COUNT(*) OVER (
                    PARTITION BY subject_id
                ) AS patient_admissions
            FROM admissions
        ) AS w
    ) AS window_output_rows;
""", title="Validate output grain")

## 30. Verify ranking tie behavior

In [ ]:
display_query("""
SELECT
    employee_name,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC, employee_id
    ) AS row_number,
    RANK() OVER (
        ORDER BY salary DESC
    ) AS rank_value,
    DENSE_RANK() OVER (
        ORDER BY salary DESC
    ) AS dense_rank_value
FROM employees
ORDER BY salary DESC, employee_id;
""", title="Validate ranking ties")

## 31. Verify that the running total reaches the source total

In [ ]:
display_query("""
WITH running AS (
    SELECT
        sale_id,
        SUM(revenue) OVER (
            ORDER BY sale_date, sale_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_revenue
    FROM sales
)
SELECT
    MAX(running_revenue) AS final_running_total,
    (SELECT SUM(revenue) FROM sales) AS source_total
FROM running;
""", title="Validate the running total")

# Part XII · Practice

These are intentionally incomplete so students can write the SQL themselves.

## Suggested way to solve each practice problem

Before writing SQL:

1. **Write the desired output grain.**  
   Example: “one row per employee” or “one row per patient.”

2. **Sketch 3–5 rows on paper.**  
   Small tables expose logic mistakes quickly.

3. **Decide whether rows should collapse.**  
   - collapse → consider `GROUP BY`
   - preserve detail → consider a window function

4. **If the problem has multiple stages, name them.**  
   That usually suggests a CTE.

5. **For sequence questions, define the order explicitly.**

6. **Run intermediate stages before the final query.**

This is the same reasoning process used in the worked examples above.

## Practice 1 · CTE

Return patients age **50 or older** and their admission counts.

Requirements:
- use a CTE for filtered patients,
- keep patients even if they have no admissions,
- return one row per qualifying patient.

<details>
<summary>Hint</summary>

Start with:

```sql
WITH older_patients AS (
    SELECT ...
    FROM patients
    WHERE ...
)
```

Then join the CTE to `admissions`.

</details>

In [ ]:
practice_1 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_1, title="Practice 1")

## Practice 2 · Ranking

Return the top **2 employees by salary inside each department**.

Try:
1. `ROW_NUMBER()`
2. `RANK()`

Compare what happens when salaries tie.

<details>
<summary>Hint</summary>

Use:

```sql
PARTITION BY department
ORDER BY salary DESC
```

Calculate the rank in a CTE and filter it in the outer query.

</details>

In [ ]:
practice_2 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_2, title="Practice 2")

## Practice 3 · Previous admission

For each repeat admission:
- show the current admission date,
- show the previous admission date,
- calculate time between admission dates.

<details>
<summary>Hint</summary>

Use `LAG(admit_time)` and partition by `subject_id`.

</details>

In [ ]:
practice_3 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_3, title="Practice 3")

## Practice 4 · Moving average

1. Aggregate sales to one row per day.
2. Calculate a three-row moving average.

<details>
<summary>Hint</summary>

```sql
ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
```

</details>

In [ ]:
practice_4 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_4, title="Practice 4")

## Practice 5 · Recursive hierarchy

Use a recursive CTE to return:
- `employee_id`
- `employee_name`
- `manager_id`
- `depth`

<details>
<summary>Hint</summary>

The anchor uses `manager_id IS NULL`.

The recursive term matches an employee's `manager_id` to the employee already found in the recursive result.

</details>

In [ ]:
practice_5 = """
-- Write your SQL here


"""

# Uncomment when ready:
# display_query(practice_5, title="Practice 5")

# Part XIII · Assessment Worked Example

### Task
Return the **two longest stays for each department**.

Requirements:
- preserve admission-level detail,
- calculate stay length,
- rank independently inside each department,
- break exact ties deterministically,
- return at most two rows per department.

In [ ]:
display_query("""
WITH ranked AS (
    SELECT
        hadm_id,
        subject_id,
        department,
        admit_time,
        discharge_time,
        EXTRACT(
            EPOCH FROM (discharge_time - admit_time)
        ) / 86400.0 AS stay_days,
        ROW_NUMBER() OVER (
            PARTITION BY department
            ORDER BY
                (discharge_time - admit_time) DESC,
                hadm_id
        ) AS rn
    FROM admissions
    WHERE discharge_time IS NOT NULL
)
SELECT *
FROM ranked
WHERE rn <= 2
ORDER BY department, rn;
""", title="Two longest stays per department")

# Part XIV · Quick Concept Check

### 1. Named multi-step logic?
<details><summary>Show answer</summary>Use a **CTE**.</details>

### 2. Parent-child hierarchy traversal?
<details><summary>Show answer</summary>Use a **recursive CTE**.</details>

### 3. Keep every admission row and add the patient's average?
<details><summary>Show answer</summary>Use a window aggregate with `PARTITION BY subject_id`.</details>

### 4. Exactly two rows per department?
<details><summary>Show answer</summary>Use `ROW_NUMBER()` and filter the generated row number.</details>

### 5. Include tied second-place salaries?
<details><summary>Show answer</summary>Use `RANK()`.</details>

### 6. Previous event?
<details><summary>Show answer</summary>Use `LAG()` with meaningful partitioning and ordering.</details>

### 7. Running calculation?
<details><summary>Show answer</summary>Use an ordered window with an explicit cumulative frame.</details>

# Module 11 Summary

### Keep these habits

1. **Name useful stages.** CTEs make multi-step logic easier to reason about.
2. **Keep grain explicit.** Know what one row represents at every stage.
3. **Windows preserve detail.** They add context without automatically collapsing rows.
4. **Order is logic.** Ranking, `LAG`, `LEAD`, and running calculations depend on it.
5. **Frames matter.** Running, moving, and `LAST_VALUE()` behavior can depend on boundaries.
6. **Choose tie behavior deliberately.** `ROW_NUMBER`, `RANK`, and `DENSE_RANK` answer different questions.

| Question | Starting pattern |
|---|---|
| Need named intermediate stages? | CTE |
| Need parent-child traversal? | `WITH RECURSIVE` |
| Need group context beside detail? | window aggregate |
| Need unique Top-N rows? | `ROW_NUMBER()` |
| Need ranking with ties? | `RANK()` / `DENSE_RANK()` |
| Need previous / next event? | `LAG()` / `LEAD()` |
| Need running or moving calculation? | ordered window + explicit frame |
| Need one row per group instead of detail? | `GROUP BY` |

## Next
**Module 12: Views & Modular Queries**